# NeuroML2

[NeuroML2](https://neuroml.org/) is an XML-based, community standard for
describing neuronal models — including detailed morphologies — in a
simulator-independent way. `braincell` reads the **morphology** of one
NeuroML2 cell into a {class}`~braincell.Morphology` through
{meth}`braincell.Morphology.from_neuroml` or {class}`braincell.io.NeuroMlReader`.

Channel, biophysical and network definitions in a NeuroML2 document are not
read — declare mechanisms with {mod}`braincell.mech` as usual (see
{doc}`../concepts/mechanisms`).

## Loading

`minimal.cell.nml` is a small cell shipped with the repository: a soma, an
axon, and a dendritic trunk in a cable group with two side branches.

In [1]:
from pathlib import Path

import brainunit as u

import braincell

path = Path("../../data/morphology/neuroml2/minimal.cell.nml")
morpho, report = braincell.Morphology.from_neuroml(path, return_report=True)
print(report)

for branch in morpho.branches:
    parent = "-" if branch.parent is None else f"{branch.parent.name} at x={float(branch.parent_x):g}"
    print(f"{branch.name:11s} {branch.type:9s} {branch.length.to_decimal(u.um):5.1f} um  parent: {parent}")

NeuroML2 report: 0 issues
soma        soma       10.0 um  parent: -
trunk_0     dendrite   25.0 um  parent: soma at x=1
trunk_1     dendrite   15.0 um  parent: trunk_0 at x=1
axon_0      axon       40.0 um  parent: soma at x=0
dendrite_0  dendrite   20.0 um  parent: soma at x=0.5
dendrite_1  dendrite   10.0 um  parent: trunk_0 at x=1


Without `return_report=True` only the morphology is returned. In a document
with several cells, choose one with `cell_id`:

```python
morpho = braincell.Morphology.from_neuroml("network.nml", cell_id="pyr")
```

`<include>` elements are not followed: the cell and its `<morphology>` must
be in the file itself (a top-level `<morphology>` referred to by
`<cell morphology="...">` is fine).

## How segments become branches

| NeuroML2 | `braincell` |
|---|---|
| a segment at the distal end of its parent (`fractionAlong="1"`, the default) | continues the parent's branch when it is the only child there, has the same type and cable group, and starts where the parent ends; otherwise it starts a new branch at `parent_x=1` |
| a change of diameter where segments join | a zero-length segment that steps between the two radii, so the area includes the annulus |
| `0 < fractionAlong < 1` | the parent's branch is cut in two at that point and the child attaches at the end of the first part |
| `fractionAlong="0"` | `parent_x=0` when the parent is the first segment of its branch; otherwise the branch is cut at the parent's start |
| `fractionAlong="0.5"` on a one-segment soma | `parent_x=0.5`, without a cut |
| omitted `<proximal>` | the point at `fractionAlong` on the parent |
| a spherical segment (proximal and distal at one point) | a soma of length and diameter equal to the sphere's diameter, centred on it, so its area is the sphere's; children attach at `parent_x=0.5`. Only a soma may be spherical |
| `soma_group`, `axon_group`, `dendrite_group`, or their NeuroLex ids | the branch type; other segments are `"custom"` |
| a cable group (`neuroLexId="sao864921383"`) | the branch name, with `_0`, `_1`, ... when the group was cut into several branches. An id that cannot name a branch (not a Python identifier, a reserved name such as `length`, or a name already taken) gives an automatic name and a `naming.group_id_unusable` warning in the report |
| other segment groups | listed in the report; they do not change the geometry |
| `<biophysicalProperties>`, `<notes>`, `<annotation>`, ... | skipped and listed in the report |

Input that is not well-formed XML, is not NeuroML2, declares a DTD, or whose
segments do not form one tree raises `ValueError` with a message that starts
`NeuroML2 import failed for <path>:` and names the element at fault.

## Segment groups and the report

braincell regions are filter expressions, so segment groups do not become
regions. The report's {class}`~braincell.io.NeuroMlMetadata` maps them onto
the branches instead: `segment_groups` lists each group's segments, and
`segment_branches` gives the part of a branch each segment covers, as an
interval of normalized length `x`.

In [2]:
metadata = report.metadata
print("cell:", metadata.cell_id, " morphology:", metadata.morphology_id)
print("groups:", metadata.segment_groups)
for segment_id, parts in metadata.segment_branches.items():
    print(f"segment {segment_id} ({metadata.segment_names[segment_id]}):", parts)

cell: minimal  morphology: minimal_morph
groups: {'soma_group': (0,), 'axon_group': (4,), 'trunk': (1, 2), 'dendrite_group': (1, 2, 3, 5), 'distal_part': (2, 3)}
segment 0 (soma): (('soma', 0.0, 1.0),)
segment 1 (trunk_a): (('trunk_0', 0.0, 0.8),)
segment 2 (trunk_b): (('trunk_0', 0.8, 1.0), ('trunk_1', 0.0, 1.0))
segment 3 (side): (('dendrite_1', 0.0, 1.0),)
segment 4 (axon): (('axon_0', 0.0, 1.0),)
segment 5 (mid): (('dendrite_0', 0.0, 1.0),)


Segment 2 holds both ends of the cut: segment 3 attaches a quarter of the way
along it, so its first quarter finishes `trunk_0` and the rest is `trunk_1`.

## Lower-level reader

{class}`~braincell.io.NeuroMlReader` takes the same arguments:

```python
from braincell.io import NeuroMlReader

morpho = NeuroMlReader().read("cell.nml", cell_id=None)
```

## See also

- {doc}`swc` and {doc}`asc` — the other morphology formats.
- {doc}`../apis/io` — the full IO reference, including
  {class}`~braincell.io.NeuroMlReport` and {class}`~braincell.io.NeuroMlIssue`.